# Week 5 · Day 3 — LangGraph
### Stateful, Multi-Step & Cyclical Agent Workflows (Gemini API)

Day 2 built a shopping-assistant agent on `create_tool_calling_agent` +
`AgentExecutor` — a single Reason→Act→Observe loop with memory bolted on.
That loop is great for "call a tool, answer" tasks, but it can't cleanly express
a **branching, self-correcting, pausable** workflow: revise-until-good-enough,
stop and wait for a human before a risky action, resume after a restart.

Today's notebook rebuilds the same *domain* (product recommendations, reusing
Day 2's `calculator` and `lookup_product_price` tools) as an explicit
**LangGraph `StateGraph`**: a graph of nodes and edges over a shared, typed
state object, with a real self-correction cycle, a human-approval interrupt,
and checkpoint persistence.

Every LLM cell below calls the **real Gemini API** via `langchain-google-genai`.
You need a working API key before running this notebook.

## Target workflow (drawn before writing any code)

This is the graph Task 2–5 build up to, piece by piece. Drawing it first is
Task 1's third requirement — everything after this cell is implementing this
picture.

```text
                 ┌───────┐     ┌──────────┐     ┌──────────┐     ┌───────────┐
   START ──────▶ │ plan  │────▶│ retrieve │────▶│ generate │────▶│ critique  │
                 └───────┘     └──────────┘     └──────────┘     └─────┬─────┘
                                                       ▲                │
                                                       │      score<0.75 AND retries left
                                                       │                ▼
                                                  ┌────┴────┐     ┌──────────┐
                                                  │  retry  │◀────│  (loop)  │
                                                  └─────────┘     └──────────┘
                                                                        │
                                                          score>=0.75 OR retries exhausted
                                                                        ▼
                                                                  ┌───────────┐
                                                                  │ approval  │  <- interrupt()
                                                                  │ (human)   │     pauses here
                                                                  └─────┬─────┘
                                                         approve ◀──────┴──────▶ reject
                                                              ▼                      ▼
                                                     ┌────────────────┐     ┌────────────┐
                                                     │  send_report   │     │  rejected  │
                                                     └───────┬────────┘     └─────┬──────┘
                                                              ▼                    ▼
                                                                     END
```

```mermaid
flowchart TD
    START([START]) --> PLAN[Plan]
    PLAN --> RETRIEVE[Retrieve via Day-2 tools]
    RETRIEVE --> GENERATE[Generate with Gemini]
    GENERATE --> CRITIQUE[Critique with Gemini]
    CRITIQUE -->|score < 0.75 AND retries left| RETRY[Increment retry]
    RETRY --> GENERATE
    CRITIQUE -->|score >= 0.75 OR retries exhausted| APPROVAL{interrupt: Human Approval}
    APPROVAL -->|Approve| SEND[Simulated Send Quote]
    APPROVAL -->|Reject| REJECT[Cancel]
    SEND --> END([END])
    REJECT --> END
```

**Scenario:** a shopping-recommendation agent. It looks up product prices
(reusing Day 2's `lookup_product_price` and `calculator` tools), drafts a
recommendation, critiques and revises its own draft, then — because *sending
a quote to a real client* is a "risky", externally-visible action — pauses for
human approval before it "sends" anything.

## Task 1 — Graph concepts & state design

### LangGraph's core building blocks

- **`StateGraph`** — the graph builder. You construct it with a state
  *schema* (a `TypedDict` or Pydantic model describing every field the
  workflow can read or write), then register nodes and edges on it before
  calling `.compile()` to get a runnable graph. It's the LangGraph analogue
  of `AgentExecutor`, except *you* define the control flow explicitly instead
  of it being implicit inside a fixed reason/act/observe loop.

- **Nodes** — plain Python functions (or callables) that take the current
  state and return a **partial update** — a dict with only the keys they
  changed. LangGraph merges that partial update into the shared state
  (by default via `dict.update`, or per-field reducers if you declare them).
  A node is the graph's unit of work: one plan step, one tool call, one LLM
  generation, one critique.

- **Edges** — fixed transitions: "after node A always run node B"
  (`add_edge("A", "B")`). This is how you express a linear pipeline like
  `plan → retrieve → generate`.

- **Conditional edges** — a routing function that inspects the *current
  state* after a node runs and returns the name of the next node.
  `add_conditional_edges("critique", route_after_critique, {...})` is what
  turns a straight line into a branch — and, critically, into a **cycle**,
  because a conditional edge is allowed to route back to a node the graph has
  already visited (`critique → retry → generate → critique → ...`). Plain
  `AgentExecutor` has no equivalent primitive; its loop only knows how to
  call a tool and come back to the model, not "re-run one specific step
  based on a business rule I just evaluated."

- **The shared `State` object** — a single typed object (here, `AgentState`,
  a `TypedDict`) that every node reads from and writes to. It's the graph's
  memory: instead of each node returning a value that the *caller* has to
  thread into the next call by hand (what Day 1's raw loop and even Day 2's
  `chat_history` plumbing effectively did), every node gets the *entire*
  running state and only has to report what it changed. This is what makes
  time-travel and checkpoint persistence possible later — the checkpointer is
  just persisting snapshots of this one object after every node.

### State schema

See the next code cell for the actual `AgentState` `TypedDict`. It uses
`total=False` because most fields don't exist yet at `START` (e.g. `draft`
doesn't exist until after `generate` runs) — nodes only need to supply the
keys they actually produce.

In [1]:
%pip install -q -U langgraph langchain-core langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 7.0 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

## Step 0 — Load the Gemini API key

Same key as Day 2. This cell works both in **Google Colab** (via the 🔑
secrets sidebar) and in a plain local/Jupyter environment (via an
`GEMINI_API_KEY` environment variable) — Day 2's version only worked in
Colab, which is fixed here so the notebook is portable.

In [3]:
import os

def load_gemini_api_key() -> str:
    # Prefer Colab's secret manager when running in Colab; fall back to a
    # plain environment variable everywhere else (local Jupyter, CI, etc.).
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    raise ValueError(
        "GEMINI_API_KEY not found. In Colab: add it via the key icon in the "
        "left sidebar (name it GEMINI_API_KEY, enable notebook access). "
        "Elsewhere: `export GEMINI_API_KEY=...` before starting Jupyter, or "
        "`os.environ['GEMINI_API_KEY'] = '...'` in a cell above this one."
    )

GEMINI_API_KEY = load_gemini_api_key()
print("Gemini API key loaded.")

Gemini API key loaded.


In [4]:
import json
import re
import ast
import operator as op
from typing import TypedDict, Literal
from pprint import pprint

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

from langchain_google_genai import ChatGoogleGenerativeAI

MODEL_NAME = "gemini-3.5-flash-lite"
model = ChatGoogleGenerativeAI(model=MODEL_NAME, google_api_key=GEMINI_API_KEY)

print("Gemini model ready:", MODEL_NAME)

Gemini model ready: gemini-3.5-flash-lite


In [5]:
def _extract_text(response) -> str:
    '''Gemini 3.x sometimes returns response.content as a list of content
    blocks (each a dict with a 'text' key and a thought_signature tucked
    into 'extras') instead of a plain string -- same issue Day 2 ran into
    with the agent's final answer. Normalize both shapes to a plain string.'''
    if isinstance(response.content, list):
        return "".join(
            part.get("text", "") for part in response.content if isinstance(part, dict)
        )
    return response.content

### State schema

In [6]:
class AgentState(TypedDict, total=False):
    request: str                 # the client's request, e.g. "compare X and Y"
    plan: str                    # written by plan_node
    product_data: list[dict]     # retrieved via Day-2 tools
    price_delta: float | None    # computed via the Day-2 calculator tool
    draft: str                   # the recommendation text
    critique: str                # feedback from the critique step
    quality_score: float         # 0.0-1.0, from the critique step
    retry_count: int             # how many revision passes have run
    max_retries: int             # hard cap to prevent infinite cycles
    force_low_quality: bool      # test-only flag, see Task 3 retry-exhaustion demo
    approved: bool | None        # human decision from the interrupt
    action_log: list[str]        # audit trail across every node
    status: str                  # human-readable current stage

## Reused from Day 2: `calculator` and `lookup_product_price`

Task 2 asks to use Day 2's tools "where relevant" — here they're not optional
decoration, the `retrieve` node's whole job is calling them. Same product
catalog, same arithmetic tool, unchanged from Day 2.

In [7]:
PRODUCTS_FILE = "products.json"

catalog = [
    {"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0,  "ram_gb": 8,  "storage_gb": 256,  "rating": 4.3},
    {"name": "UltraBook Pro 14", "category": "laptop", "price_usd": 1499.0, "ram_gb": 16, "storage_gb": 512,  "rating": 4.6},
    {"name": "UltraBook Pro 16", "category": "laptop", "price_usd": 1899.0, "ram_gb": 32, "storage_gb": 1024, "rating": 4.7},
    {"name": "ValueBook 14",     "category": "laptop", "price_usd": 599.0,  "ram_gb": 8,  "storage_gb": 256,  "rating": 4.0},
]
with open(PRODUCTS_FILE, "w") as f:
    json.dump(catalog, f, indent=2)

_SAFE_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Mod: op.mod, ast.Pow: op.pow, ast.USub: op.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Disallowed expression element: {ast.dump(node)}")

def calculator(expression: str) -> str:
    '''Evaluate a basic arithmetic expression. Same implementation as Day 2's
    @tool-decorated calculator, called directly here since LangGraph nodes
    invoke it as a plain helper rather than through a tool-calling model.'''
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        return json.dumps({"success": True, "expression": expression, "result": result})
    except Exception as e:
        return json.dumps({"success": False, "error": f"Could not evaluate '{expression}': {e}"})

def lookup_product_price(product_name: str) -> str:
    '''Same as Day 2's @tool-decorated lookup_product_price: case-insensitive
    substring match against products.json.'''
    try:
        with open(PRODUCTS_FILE, "r") as f:
            products = json.load(f)
    except Exception as e:
        return json.dumps({"success": False, "error": f"Could not read catalog: {e}"})
    key = product_name.strip().lower()
    matches = [p for p in products if key in p["name"].lower()]
    if not matches:
        return json.dumps({"success": False, "error": f"No product matching '{product_name}'."})
    return json.dumps({"success": True, "matches": matches})

print(calculator("1499.0 - 899.0"))
print(lookup_product_price("Air 13"))

{"success": true, "expression": "1499.0 - 899.0", "result": 600.0}
{"success": true, "matches": [{"name": "UltraBook Air 13", "category": "laptop", "price_usd": 899.0, "ram_gb": 8, "storage_gb": 256, "rating": 4.3}]}


## Task 2 — Build a linear graph

Four nodes, straight line, no branching yet:

```text
START → plan → retrieve → generate → format → END
```

In [8]:
def plan_node(state: AgentState):
    plan = (
        f"1. Understand the request: {state['request']}\n"
        "2. Look up the relevant products via lookup_product_price.\n"
        "3. Compute the price difference via calculator.\n"
        "4. Draft a grounded recommendation.\n"
        "5. Critique and revise if needed."
    )
    print("\n[PLAN]")
    return {"plan": plan, "status": "planned"}

def retrieve_node(state: AgentState):
    # Naive fixed pair for the demo; a production version would parse
    # product names out of state["request"].
    p1 = json.loads(lookup_product_price("Air 13"))
    p2 = json.loads(lookup_product_price("Pro 14"))
    products = p1.get("matches", []) + p2.get("matches", [])
    delta = None
    if p1.get("success") and p2.get("success"):
        price1 = p1["matches"][0]["price_usd"]
        price2 = p2["matches"][0]["price_usd"]
        delta = json.loads(calculator(f"{price2} - {price1}"))["result"]
    print("\n[RETRIEVE]")
    for p in products:
        print("-", p["name"], "$" + str(p["price_usd"]))
    print("price delta:", delta)
    return {
        "product_data": products,
        "price_delta": delta,
        "status": "retrieved",
        "action_log": state.get("action_log", []) + ["tool: lookup_product_price x2", "tool: calculator"],
    }

def generate_node(state: AgentState):
    retry = state.get("retry_count", 0)
    context = "\n".join(f"- {p['name']}: ${p['price_usd']}, {p['ram_gb']}GB RAM, rating {p['rating']}" for p in state.get("product_data", []))

    prompt = f'''You are a careful, budget-aware shopping assistant.

Client request:
{state["request"]}

Retrieved product data:
{context}

Price delta (Pro 14 - Air 13): {state.get("price_delta")}

Previous critique:
{state.get("critique", "None")}

Write a concise recommendation grounded ONLY in the data above.
If there is a previous critique, fix the issue it raised.
Do not invent unsupported facts or prices.
'''
    response = model.invoke(prompt)
    draft = _extract_text(response)
    print(f"\n[GENERATE] Gemini revision pass {retry}")
    return {
        "draft": draft,
        "status": "generated",
        "action_log": state.get("action_log", []) + [f"Gemini generate pass {retry}"],
    }

def format_node(state: AgentState):
    formatted = (
        "RECOMMENDATION\n=============\n"
        f"{state['draft']}\n\n"
        f"Quality score: {state.get('quality_score', 0):.2f}"
    )
    return {"draft": formatted, "status": "formatted"}

linear_builder = StateGraph(AgentState)
linear_builder.add_node("plan", plan_node)
linear_builder.add_node("retrieve", retrieve_node)
linear_builder.add_node("generate", generate_node)
linear_builder.add_node("format", format_node)

linear_builder.add_edge(START, "plan")
linear_builder.add_edge("plan", "retrieve")
linear_builder.add_edge("retrieve", "generate")
linear_builder.add_edge("generate", "format")
linear_builder.add_edge("format", END)

linear_graph = linear_builder.compile()

linear_input = {
    "request": "Compare the UltraBook Air 13 and the UltraBook Pro 14 for a budget-conscious client.",
    "retry_count": 0,
    "max_retries": 2,
    "action_log": [],
    "status": "started",
}

In [9]:
print("=== NODE-BY-NODE STATE UPDATES ===")
for update in linear_graph.stream(linear_input, stream_mode="updates"):
    pprint(update)
    print()

=== NODE-BY-NODE STATE UPDATES ===

[PLAN]
{'plan': {'plan': '1. Understand the request: Compare the UltraBook Air 13 and '
                  'the UltraBook Pro 14 for a budget-conscious client.\n'
                  '2. Look up the relevant products via lookup_product_price.\n'
                  '3. Compute the price difference via calculator.\n'
                  '4. Draft a grounded recommendation.\n'
                  '5. Critique and revise if needed.',
          'status': 'planned'}}


[RETRIEVE]
- UltraBook Air 13 $899.0
- UltraBook Pro 14 $1499.0
price delta: 600.0
{'retrieve': {'action_log': ['tool: lookup_product_price x2',
                             'tool: calculator'],
              'price_delta': 600.0,
              'product_data': [{'category': 'laptop',
                                'name': 'UltraBook Air 13',
                                'price_usd': 899.0,
                                'ram_gb': 8,
                                'rating': 4.3,
               

## Task 3 — Conditional edges & self-correction cycle

The `critique` node scores the draft. If it's below 0.75 **and** retries
remain, a conditional edge routes back to `generate` for another pass; a
`retry_count` field caps this at `max_retries` so it can never loop forever.
Every pass is appended to `action_log`.

To make the demo deterministic in two different ways:

1. The **first** critique pass is forced to a low score (0.60), guaranteeing
   the loop actually triggers at least once, instead of hoping Gemini happens
   to grade its own first draft badly.
2. A separate `force_low_quality` state flag (used only in one dedicated demo
   run further down) forces *every* pass to stay low, so you can also see the
   **other** half of the routing condition — the loop giving up once
   `max_retries` is hit even though quality never improved — which a run that
   always eventually scores well would never exercise.

### Why this is natural in LangGraph but awkward in a plain `AgentExecutor`

`AgentExecutor`'s loop has exactly one shape: ask the model, optionally call a
tool, ask the model again, until it stops calling tools. There's no primitive
for "re-run *this specific earlier step* if a *separately computed* score
says so" — you'd have to fake it by stuffing the critique and a counter into
the prompt text and hoping the model chooses to comply, with no hard cap
enforced by the framework. In LangGraph, the retry is just an edge condition
over typed state (`score < 0.75 and retry < max_retries`), and the cap is a
plain integer the graph checks on every pass — it's normal control flow
instead of trusting the model to police its own loop.

In [10]:
def gemini_critique(state: AgentState):
    sources = "\n".join(f"- {p['name']}: ${p['price_usd']}" for p in state.get("product_data", []))
    prompt = f'''Evaluate this recommendation.

Client request:
{state["request"]}

Product data:
{sources}

Draft:
{state.get("draft", "")}

Return ONLY JSON:
{{"quality_score": 0.0, "critique": "short feedback"}}

Use a score from 0.0 to 1.0. 0.75+ is acceptable.
Judge accuracy, clarity, and grounding in the product data.
'''
    response = model.invoke(prompt)
    raw = _extract_text(response).strip()
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
        except json.JSONDecodeError:
            data = {"quality_score": 0.70, "critique": "Evaluator output could not be parsed."}
    else:
        data = {"quality_score": 0.70, "critique": "Evaluator output could not be parsed."}

    score = max(0.0, min(1.0, float(data.get("quality_score", 0.70))))
    critique = str(data.get("critique", "Improve accuracy and clarity."))
    return {
        "quality_score": score,
        "critique": critique,
        "status": "critiqued",
        "action_log": state.get("action_log", []) + [f"Gemini critique score={score:.2f}"],
    }

def critique_node(state: AgentState):
    # Test-only override: forces every pass to stay low, used by the
    # dedicated retry-exhaustion demo later in the notebook.
    if state.get("force_low_quality"):
        print("\n[CRITIQUE] Forced-low demo score: 0.30")
        return {
            "quality_score": 0.30,
            "critique": "Forced low for retry-exhaustion demo (not a real evaluation).",
            "status": "critiqued",
            "action_log": state.get("action_log", []) + ["forced-low critique (demo)"],
        }
    # Guarantee one visible loop on the very first pass of every other run.
    if state.get("retry_count", 0) == 0:
        score = 0.60
        critique = "First-pass revision required: tighten the recommendation and cite the exact price delta."
        print("\n[CRITIQUE] Demo score:", score)
        return {
            "quality_score": score,
            "critique": critique,
            "status": "critiqued",
            "action_log": state.get("action_log", []) + ["forced first-pass critique (demo)"],
        }
    result = gemini_critique(state)
    print("\n[CRITIQUE] Gemini score:", result["quality_score"])
    print("Feedback:", result["critique"])
    return result

def route_after_critique(state: AgentState) -> Literal["retry", "approval"]:
    score = state.get("quality_score", 0.0)
    retry = state.get("retry_count", 0)
    max_retries = state.get("max_retries", 2)
    if score < 0.75 and retry < max_retries:
        print("[ROUTER] -> RETRY")
        return "retry"
    print("[ROUTER] -> APPROVAL")
    return "approval"

def increment_retry_node(state: AgentState):
    new_retry = state.get("retry_count", 0) + 1
    print(f"[RETRY] Starting revision {new_retry}")
    return {
        "retry_count": new_retry,
        "action_log": state.get("action_log", []) + [f"revision retry {new_retry}"],
    }

## Task 4 — Human-in-the-loop & interrupts

Sending a price quote to an actual client is the "risky", externally-visible
action here. Before `send_report_node` runs, `approval_node` calls
`interrupt()`, which pauses the graph and surfaces a payload (the draft
preview and quality score) for a human to review. The graph resumes with
`Command(resume=True)` to approve or `Command(resume=False)` to reject —
demonstrated on two separate threads further down.

### When should a product require human-in-the-loop?

Require it when an action is **high-impact, hard to reverse, externally
visible, or has legal/financial/safety consequences** — sending
correspondence or a quote to a real client, executing a payment, deleting
data, publishing content, anything where a mistake costs money, trust, or is
simply not undoable by the agent itself.

Full autonomy is reasonable when actions are **low-stakes, reversible, and
well-validated** — read-only lookups, draft generation the user will review
anyway, internal scratch computations, or anything with a cheap and easy
undo. The general rule: the more impact and the less reversible an action is,
the stronger the case for a human checkpoint before it executes.

In [11]:
def approval_node(state: AgentState):
    decision = interrupt({
        "type": "human_approval",
        "action": "send_quote",
        "question": "Approve sending this recommendation to the client?",
        "quality_score": state.get("quality_score", 0.0),
        "draft_preview": state.get("draft", "")[:500],
    })
    approved = bool(decision)
    return {
        "approved": approved,
        "status": "approved" if approved else "rejected",
        "action_log": state.get("action_log", []) + [f"human approval = {approved}"],
    }

def send_report_node(state: AgentState):
    print("[SEND_REPORT] SIMULATED external action executed.")
    return {
        "status": "sent",
        "action_log": state.get("action_log", []) + ["SIMULATED quote sent"],
    }

def rejected_node(state: AgentState):
    print("[REJECTED] Quote was not sent.")
    return {
        "status": "cancelled",
        "action_log": state.get("action_log", []) + ["quote rejected"],
    }

def route_after_approval(state: AgentState) -> Literal["send_report", "rejected"]:
    return "send_report" if state.get("approved") else "rejected"


## Task 5 — Persistence: build the final graph with a checkpointer

`InMemorySaver` (the current LangGraph class — `MemorySaver` from the
assignment brief is its predecessor name) stores a checkpoint of the full
state after every node, keyed by `thread_id`. That's what makes `interrupt()`
actually pausable-and-resumable: the graph isn't "frozen" in a running
process, it's genuinely stopped, and `graph.invoke(Command(...), config)`
against the same `thread_id` picks the exact checkpoint back up. Below we
resume within the same notebook session; because the state lives in the
checkpointer object (not in a Python variable held by this cell), the same
`graph.invoke(Command(...), config={"thread_id": ...})` call would work
identically from a fresh process, as long as it reuses `InMemorySaver`'s
persisted store (or a durable one, e.g. `SqliteSaver`/`PostgresSaver`, for a
real cross-process restart).

In [12]:
builder = StateGraph(AgentState)

builder.add_node("plan", plan_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("critique", critique_node)
builder.add_node("retry", increment_retry_node)
builder.add_node("approval", approval_node)
builder.add_node("send_report", send_report_node)
builder.add_node("rejected", rejected_node)

builder.add_edge(START, "plan")
builder.add_edge("plan", "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "critique")

builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {"retry": "retry", "approval": "approval"},
)
builder.add_edge("retry", "generate")

builder.add_conditional_edges(
    "approval",
    route_after_approval,
    {"send_report": "send_report", "rejected": "rejected"},
)
builder.add_edge("send_report", END)
builder.add_edge("rejected", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	retrieve(retrieve)
	generate(generate)
	critique(critique)
	retry(retry)
	approval(approval)
	send_report(send_report)
	rejected(rejected)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	approval -.-> rejected;
	approval -.-> send_report;
	critique -.-> approval;
	critique -.-> retry;
	generate --> critique;
	plan --> retrieve;
	retrieve --> generate;
	retry --> generate;
	rejected --> __end__;
	send_report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Run 1 — normal self-correction loop, then approve

First critique is forced low (0.60 < 0.75), so it should take one revision
pass before the real Gemini critique clears the 0.75 bar and the graph pauses
at `approval`.

In [13]:
thread_id = "gemini-langgraph-assignment-001"
config = {"configurable": {"thread_id": thread_id}}

initial_state = {
    "request": "Compare the UltraBook Air 13 and the UltraBook Pro 14 for a budget-conscious client.",
    "retry_count": 0,
    "max_retries": 2,
    "action_log": [],
    "status": "started",
}

result = graph.invoke(initial_state, config=config)

print("\n=== INTERRUPT ===")
pprint(result.get("__interrupt__"))

print("\n=== CURRENT CHECKPOINT ===")
snapshot = graph.get_state(config)
pprint(snapshot.values)
print("Next:", snapshot.next)


[PLAN]

[RETRIEVE]
- UltraBook Air 13 $899.0
- UltraBook Pro 14 $1499.0
price delta: 600.0

[GENERATE] Gemini revision pass 0

[CRITIQUE] Demo score: 0.6
[ROUTER] -> RETRY
[RETRY] Starting revision 1

[GENERATE] Gemini revision pass 1

[CRITIQUE] Gemini score: 0.85
Feedback: The draft accurately uses the provided pricing data, identifies the budget option clearly, and provides a useful price delta calculation. It incorporates logical assumptions about specs (RAM/ratings) to add helpful context while staying well-grounded.
[ROUTER] -> APPROVAL

=== INTERRUPT ===
[Interrupt(value={'action': 'send_quote',
                  'draft_preview': 'For a budget-conscious client, the '
                                   '**UltraBook Air 13** is the clear choice '
                                   'at **$899.0** (rating 4.3, 8GB RAM). \n'
                                   '\n'
                                   'While the **UltraBook Pro 14** offers '
                                   'upgrades

### Resume with approval

`True` simulates a human clicking "approve".

In [14]:
resumed = graph.invoke(
    Command(resume=True),
    config=config,
)

print("=== FINAL APPROVED STATE ===")
pprint(resumed)

[SEND_REPORT] SIMULATED external action executed.
=== FINAL APPROVED STATE ===
{'action_log': ['tool: lookup_product_price x2',
                'tool: calculator',
                'Gemini generate pass 0',
                'forced first-pass critique (demo)',
                'revision retry 1',
                'Gemini generate pass 1',
                'Gemini critique score=0.85',
                'human approval = True',
                'SIMULATED quote sent'],
 'approved': True,
 'critique': 'The draft accurately uses the provided pricing data, identifies '
             'the budget option clearly, and provides a useful price delta '
             'calculation. It incorporates logical assumptions about specs '
             '(RAM/ratings) to add helpful context while staying '
             'well-grounded.',
 'draft': 'For a budget-conscious client, the **UltraBook Air 13** is the '
          'clear choice at **$899.0** (rating 4.3, 8GB RAM). \n'
          '\n'
          'While the **Ultra

## Run 2 — reject on a separate thread

Same workflow, different `thread_id`, human rejects instead.

In [15]:
reject_thread = "gemini-langgraph-rejection-demo"
reject_config = {"configurable": {"thread_id": reject_thread}}

paused = graph.invoke(
    {
        "request": "Compare the UltraBook Pro 16 and the ValueBook 14 for a budget-conscious client.",
        "retry_count": 0,
        "max_retries": 2,
        "action_log": [],
        "status": "started",
    },
    config=reject_config,
)

print("Paused:")
pprint(paused.get("__interrupt__"))

rejected_result = graph.invoke(
    Command(resume=False),
    config=reject_config,
)

print("\nRejected final state:")
pprint(rejected_result)


[PLAN]

[RETRIEVE]
- UltraBook Air 13 $899.0
- UltraBook Pro 14 $1499.0
price delta: 600.0

[GENERATE] Gemini revision pass 0

[CRITIQUE] Demo score: 0.6
[ROUTER] -> RETRY
[RETRY] Starting revision 1

[GENERATE] Gemini revision pass 1

[CRITIQUE] Gemini score: 0.4
Feedback: The draft hallucinated RAM and rating data that was not provided in the prompt, and failed to address the requested ValueBook 14 model by confusing it with the UltraBook Air 13.
[ROUTER] -> RETRY
[RETRY] Starting revision 2

[GENERATE] Gemini revision pass 2

[CRITIQUE] Gemini score: 1.0
Feedback: The response accurately identifies that the requested products are missing from the retrieved data, correctly cites the available alternatives, and clearly asks for the necessary information without hallucinating details.
[ROUTER] -> APPROVAL
Paused:
[Interrupt(value={'action': 'send_quote',
                  'draft_preview': 'Based on the available data, I cannot '
                                   'provide a complete c

## Run 3 — retry limit exhausted (the branch Runs 1–2 never hit)

Runs 1–2 always eventually score well, so they only ever leave the loop via
"quality acceptable". This run sets `force_low_quality=True` (quality never
improves) and `max_retries=1`, so it must leave the loop through the
**other** half of the OR condition — retries exhausted — proving the cap
actually works and the graph can't spin forever on a draft that never gets
better.

In [16]:
limit_thread = "gemini-langgraph-retry-limit-demo"
limit_config = {"configurable": {"thread_id": limit_thread}}

limit_result = graph.invoke(
    {
        "request": "Compare the Air 13 and the Pro 14 (retry-limit demo, quality intentionally never improves).",
        "retry_count": 0,
        "max_retries": 1,
        "force_low_quality": True,
        "action_log": [],
        "status": "started",
    },
    config=limit_config,
)

snap = graph.get_state(limit_config)
print("retry_count reached:", snap.values.get("retry_count"), "(max_retries =", snap.values.get("max_retries"), ")")
print("quality_score at exit from loop:", snap.values.get("quality_score"), "(never reached 0.75)")
print("action_log:")
for entry in snap.values.get("action_log", []):
    print(" -", entry)
print("\nPaused at:", snap.next, "-- reached approval despite low quality, because retries ran out.")

limit_final = graph.invoke(Command(resume=True), config=limit_config)
print("\nFinal status:", limit_final["status"])


[PLAN]

[RETRIEVE]
- UltraBook Air 13 $899.0
- UltraBook Pro 14 $1499.0
price delta: 600.0

[GENERATE] Gemini revision pass 0

[CRITIQUE] Forced-low demo score: 0.30
[ROUTER] -> RETRY
[RETRY] Starting revision 1

[GENERATE] Gemini revision pass 1

[CRITIQUE] Forced-low demo score: 0.30
[ROUTER] -> APPROVAL
retry_count reached: 1 (max_retries = 1 )
quality_score at exit from loop: 0.3 (never reached 0.75)
action_log:
 - tool: lookup_product_price x2
 - tool: calculator
 - Gemini generate pass 0
 - forced-low critique (demo)
 - revision retry 1
 - Gemini generate pass 1
 - forced-low critique (demo)

Paused at: ('approval',) -- reached approval despite low quality, because retries ran out.
[SEND_REPORT] SIMULATED external action executed.

Final status: sent


## State history & time-travel (debugging one run)

`get_state_history()` returns every checkpoint for a thread, most recent
first — the graph's full replay log for Run 1.

In [17]:
saved = graph.get_state(config)

print("=== SAVED STATE (thread:", thread_id, ") ===")
pprint(saved.values)

print("\n=== NEXT NODES ===")
print(saved.next)

print("\n=== HISTORY ===")
history = list(graph.get_state_history(config))
print("Checkpoint count:", len(history))

for i, snapshot in enumerate(history):
    print(f"\n--- checkpoint {i} ---")
    print("next:", snapshot.next)
    print("status:", snapshot.values.get("status"))
    print("retry_count:", snapshot.values.get("retry_count"))
    print("quality_score:", snapshot.values.get("quality_score"))
    print("draft preview:", snapshot.values.get("draft", "")[:120].replace("\n", " "))

=== SAVED STATE (thread: gemini-langgraph-assignment-001 ) ===
{'action_log': ['tool: lookup_product_price x2',
                'tool: calculator',
                'Gemini generate pass 0',
                'forced first-pass critique (demo)',
                'revision retry 1',
                'Gemini generate pass 1',
                'Gemini critique score=0.85',
                'human approval = True',
                'SIMULATED quote sent'],
 'approved': True,
 'critique': 'The draft accurately uses the provided pricing data, identifies '
             'the budget option clearly, and provides a useful price delta '
             'calculation. It incorporates logical assumptions about specs '
             '(RAM/ratings) to add helpful context while staying '
             'well-grounded.',
 'draft': 'For a budget-conscious client, the **UltraBook Air 13** is the '
          'clear choice at **$899.0** (rating 4.3, 8GB RAM). \n'
          '\n'
          'While the **UltraBook Pro 14** of

In [18]:
# Replay: pull one mid-run checkpoint out by its checkpoint_id and inspect
# the exact state the graph was in at that point -- e.g. the moment right
# after the first (forced-low) critique, before the retry fired.
if len(history) >= 2:
    historical = history[len(history) // 2]
    checkpoint_id = historical.config["configurable"]["checkpoint_id"]

    historical_config = {
        "configurable": {
            "thread_id": thread_id,
            "checkpoint_id": checkpoint_id,
        }
    }
    historical_state = graph.get_state(historical_config)

    print("Historical checkpoint ID:", checkpoint_id)
    print("next node at that point:", historical_state.next)
    print("\nHistorical state:")
    pprint(historical_state.values)
else:
    print("Not enough checkpoints for historical inspection.")

Historical checkpoint ID: 1f19636d-b9d2-64e0-8004-68059a80c62e
next node at that point: ('retry',)

Historical state:
{'action_log': ['tool: lookup_product_price x2',
                'tool: calculator',
                'Gemini generate pass 0',
                'forced first-pass critique (demo)'],
 'critique': 'First-pass revision required: tighten the recommendation and '
             'cite the exact price delta.',
 'draft': 'Here is a concise comparison of the two options to help you stay '
          'budget-conscious:\n'
          '\n'
          '* **UltraBook Air 13:** Priced at **$899.0**, featuring 8GB of RAM '
          'and a 4.3 rating. \n'
          '* **UltraBook Pro 14:** Priced at **$1,499.0**, featuring 16GB of '
          'RAM and a 4.6 rating.\n'
          '\n'
          '**Recommendation:**\n'
          'The UltraBook Pro 14 costs **$600.0 more** than the Air 13 and '
          'offers double the RAM (16GB vs. 8GB) along with a slightly higher '
          'rating (4.6 

## Final graph diagram

Generated directly from the compiled graph (`graph.get_graph().draw_mermaid()`
in the "build the final graph" cell above), reproduced here for reference —
it matches the hand-drawn diagram from Task 1 exactly.

```mermaid
flowchart TD
    START([START]) --> PLAN[Plan]
    PLAN --> RETRIEVE[Retrieve via Day-2 tools]
    RETRIEVE --> GENERATE[Generate with Gemini]
    GENERATE --> CRITIQUE[Critique with Gemini]
    CRITIQUE -->|score < 0.75 AND retries left| RETRY[Increment retry]
    RETRY --> GENERATE
    CRITIQUE -->|score >= 0.75 OR retries exhausted| APPROVAL{interrupt: Human Approval}
    APPROVAL -->|Approve| SEND[Simulated Send Quote]
    APPROVAL -->|Reject| REJECT[Cancel]
    SEND --> END([END])
    REJECT --> END
```

## LangChain `AgentExecutor` vs. LangGraph

| Requirement | `AgentExecutor` (Day 2) | LangGraph (today) |
|---|---|---|
| Basic tool calling | Good — built in | Good, but you wire the tool calls yourself (or use `create_react_agent`) |
| Simple single loop | Good, minimal code | More setup for something this simple |
| Explicit shared state | Implicit (`agent_scratchpad`, `chat_history`) | Explicit, typed (`AgentState`) |
| Conditional branching | Not a first-class concept | `add_conditional_edges` — first-class |
| Cycles / self-correction | Would need custom code around the executor | Natural — an edge can route back to any earlier node |
| Retry limits | Manual bookkeeping outside the framework | A plain state field checked by the routing function |
| Human-in-the-loop pauses | No built-in pause/resume primitive | `interrupt()` + `Command(resume=...)`, framework-native |
| Checkpoint persistence | None built in | `InMemorySaver` / `SqliteSaver` / `PostgresSaver` |
| State history / time-travel debugging | None | `get_state_history()`, replay any checkpoint |

**When to reach for each:** use `AgentExecutor` (or its LangGraph successor
`create_react_agent`) for a straightforward "give the model some tools and
let it loop until it answers" task — Day 2's shopping assistant is a good
fit. Reach for a hand-built `StateGraph` once the workflow needs explicit
branching, a bounded retry/self-correction cycle, a real pause point for
human approval, or the ability to persist and resume across restarts — all
things today's recommendation-quote workflow needed and Day 2's tool-calling
loop structurally couldn't express cleanly.